**Set environment**

In [1]:
suppressMessages(suppressWarnings(source("../run_config_project.R")))
show_env()

BASE DIRECTORY (FD_BASE): /hpc/group/igvf/kk319 
REPO DIRECTORY (FD_REPO): /hpc/group/igvf/kk319/repo 
WORK DIRECTORY (FD_WORK): /hpc/group/igvf/kk319/work 
DATA DIRECTORY (FD_DATA): /hpc/group/igvf/kk319/data 

You are working with      IGVF BlueSTARR 
PATH OF PROJECT (FD_PRJ): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR 
PROJECT RESULTS (FD_RES): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results 
PROJECT SCRIPTS (FD_EXE): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts 
PROJECT DATA    (FD_DAT): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data 
PROJECT NOTE    (FD_NBK): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks 
PROJECT DOCS    (FD_DOC): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs 
PROJECT LOG     (FD_LOG): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log 
PROJECT REF     (FD_REF): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references 



## Import data

In [2]:
### set file directory
txt_prefix = "variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs"
txt_folder = "motifcount_pilot_jvierstra_v2.1beta"
txt_fdiry  = file.path(FD_RES, "analysis_variant_motif_richard", txt_folder)
txt_fname  = paste(txt_prefix, "pilot.motif.count.tsv", sep = ".")
txt_fpath  = file.path(txt_fdiry, txt_fname)

### read table
dat = read_tsv(txt_fpath, show_col_types = FALSE)

### assign and show
dat_motif_count_import = dat
print(dim(dat))
fun_display_table(head(dat))

[1] 2516    7


Group_Type,Group,Type,Motif_Name,Count_Total,Count_Gain,Count_Loss
Top_Ori,Top,Ori,AC0001:GATA/PROP:GATA,99999,361,261
Top_Ori,Top,Ori,AC0002:PROP/ALX:Homeodomain,99999,812,591
Top_Ori,Top,Ori,AC0003:HNF1A/HNF1B:Homeodomain,99999,429,466
Top_Ori,Top,Ori,AC0004:ZSCAN:C2H2_ZF,99999,185,255
Top_Ori,Top,Ori,"AC0005:POU3F/POU1F:Homeodomain,POU",99999,685,1027
Top_Ori,Top,Ori,AC0006:MEOX:Homeodomain,99999,631,783


**Check table**

In [3]:
dat = dat_motif_count_import
res = table(dat$Group_Type, dat$Group, dat$Type)
print(res)

, ,  = Nuc

         
          Low Top
  Low_Nuc 629   0
  Low_Ori   0   0
  Top_Nuc   0 629
  Top_Ori   0   0

, ,  = Ori

         
          Low Top
  Low_Nuc   0   0
  Low_Ori 629   0
  Top_Nuc   0   0
  Top_Ori   0 629



## Arrange tables

**Split into list for looping**

In [4]:
dat = dat_motif_count_import 
dat = dat %>% tidyr::pivot_longer(
    cols      = c(Count_Gain, Count_Loss),
    names_to  = "Direction",
    values_to = "Count"
)
lst = split(dat, dat$Direction)

### assign and show
lst_dat_motif_count = lst
res = lapply(lst, dim)
print(res)

$Count_Gain
[1] 2516    7

$Count_Loss
[1] 2516    7



**Run logistic regression**

In [5]:
lst = lst_dat_motif_count 
lst = lapply(lst, function(dat){
    ### split by motifs
    lst = split(dat, dat$Motif_Name)

    ### loop through motifs and run logistic regression
    lst = lapply(lst, function(dat){
        ### ensure factor level
        dat = dat %>% dplyr::mutate(
            Group = factor(Group, levels = c("Low","Top")),
            Type  = factor(Type,  levels = c("Nuc","Ori"))
        )
        
        ### pseudo-count
        dat$Count = dat$Count + 0.5
        dat$Count_Total = dat$Count_Total + 1
        
        ### sanity check
        stopifnot(all(dat$Count <= dat$Count_Total))

        ### run logistic regression
        fit = suppressWarnings(glm(
            cbind(Count, Count_Total - Count) ~ Group * Type,
            family = binomial,
            data = dat
        ))
        return(fit)
    })
    return(lst)
})

lol_fit_motif_count = lst
res = lapply(lst, length)
print(res)

$Count_Gain
[1] 629

$Count_Loss
[1] 629



**Arrange the estimation results**

In [6]:
lol_fit = lol_fit_motif_count
lst_res = lapply(names(lol_fit), function(txt_direction){

    ### get the list of regression fit for all motifs
    lst_fit = lol_fit[[txt_direction]]

    ### get coefficient of each fit model
    lst_res_direction = lapply(names(lst_fit), function(txt_motif){

        ### get the fit of a motif and summarize
        fit = lst_fit[[txt_motif]]
        res = summary(fit)$coefficients

        ### get coefficient
        txt = "GroupTop:TypeOri"
        num_beta_est  = res[txt, "Estimate"]
        num_beta_se   = res[txt, "Std. Error"]
        num_beta_pval = res[txt, "Pr(>|z|)"]

        ### arrange result table
        dat = data.frame(
            Direction = txt_direction, # "Count_Gain" or "Count_Loss"
            Motif_Name = txt_motif,    
            Beta_Est   = num_beta_est,
            OddsRatio  = exp(num_beta_est),
            Beta_SE    = num_beta_se,
            Beta_Z     = num_beta_est / num_beta_se,
            Beta_Pval  = num_beta_pval
        )
        return(dat)
    })

    ### concat results of a direction
    dat_direction = dplyr::bind_rows(lst_res_direction)
    
    ### adjust pvalue (BH) within direction
    dat_direction = dat_direction %>% 
        dplyr::mutate(
            Beta_Padj = p.adjust(Beta_Pval, method = "BH")
        )
    return(dat_direction)
})

### concat results across for both direction
dat = dplyr::bind_rows(lst_res)

### assign and show
dat_fit_motif_enrich = dat
print(dim(dat))
fun_display_table(head(dat))

[1] 1258    8


Direction,Motif_Name,Beta_Est,OddsRatio,Beta_SE,Beta_Z,Beta_Pval,Beta_Padj
Count_Gain,AC0001:GATA/PROP:GATA,0.1921645,1.2118699,0.0928337,2.069986,0.0384537,0.0666318
Count_Gain,AC0002:PROP/ALX:Homeodomain,0.8336586,2.3017246,0.0703133,11.856343,0.0000000,0.0000000
Count_Gain,AC0003:HNF1A/HNF1B:Homeodomain,0.0813782,1.0847811,0.0761162,1.069131,0.2850106,0.3719329
Count_Gain,AC0004:ZSCAN:C2H2_ZF,-0.2531942,0.7763171,0.1324355,-1.911830,0.0558980,0.0922831
Count_Gain,"AC0005:POU3F/POU1F:Homeodomain,POU",-0.1404850,0.8689367,0.0655264,-2.143943,0.0320374,0.0567649
Count_Gain,AC0006:MEOX:Homeodomain,0.2092501,1.2327532,0.0696715,3.003379,0.0026700,0.0058321


**Explore results**

In [7]:
dat = dat_fit_motif_enrich

vec1 = dat$Direction
res = table(vec1)
print(res)
cat("\n")

vec2 = ifelse(dat$Beta_Padj < 0.001, "Signif", "NotSignif")
res = table(vec1, vec2)
print(res)

vec1
Count_Gain Count_Loss 
       629        629 

            vec2
vec1         NotSignif Signif
  Count_Gain       395    234
  Count_Loss       380    249


**Arrange table columns**

In [8]:
dat = dat_fit_motif_enrich
eps = min(dat$Beta_Padj[dat$Beta_Padj > 0]) / 10

dat = dat %>% 
    dplyr::mutate(Log10Beta  = Beta_Est / log(10)) %>%
    dplyr::mutate(Beta_Padj  = pmax(Beta_Padj, eps)) %>%
    dplyr::mutate(NLog10Padj = -log10(Beta_Padj))

dat_fit_motif_enrich_arrange = dat
print(dim(dat))
fun_display_table(head(dat, 3))

[1] 1258   10


Direction,Motif_Name,Beta_Est,OddsRatio,Beta_SE,Beta_Z,Beta_Pval,Beta_Padj,Log10Beta,NLog10Padj
Count_Gain,AC0001:GATA/PROP:GATA,0.1921645,1.211870,0.0928337,2.069986,0.0384537,0.0666318,0.0834560,1.1763184
Count_Gain,AC0002:PROP/ALX:Homeodomain,0.8336586,2.301725,0.0703133,11.856343,0.0000000,0.0000000,0.3620533,30.4065576
Count_Gain,AC0003:HNF1A/HNF1B:Homeodomain,0.0813782,1.084781,0.0761162,1.069131,0.2850106,0.3719329,0.0353421,0.4295354


## Export results

In [9]:
txt_prefix = "variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs"
txt_folder = "motifcount_pilot_jvierstra_v2.1beta"
txt_fdiry  = file.path(FD_RES, "analysis_variant_motif_richard", txt_folder)
txt_fname  = paste(txt_prefix, "pilot.motif.enrich.tsv", sep = ".")
txt_fpath  = file.path(txt_fdiry, txt_fname)

dat = dat_fit_motif_enrich_arrange
write_tsv(dat, txt_fpath)